## Gathering Alpha Vantage API Data


##  Creating api function


In [0]:
import requests
import pandas as pd
import pyspark.sql.functions as F

In [0]:
%sql
USE CATALOG "alpha_vantage";
USE SCHEMA bronze;

In [0]:
def api_request(ticker):
    url = "https://www.alphavantage.co/query"
    querystring = {
        "function":"TIME_SERIES_DAILY",
        "symbol": ticker,
        "datatype":"json",
        "outputsize":"compact",
        "apikey":  '3SXVOC66SF4SEMZ2'
    }

    try:
        response = requests.get(url, params=querystring)
        response.raise_for_status()  # raises HTTPError on 4xx/5xx
        data = response.json()
        data = data["Time Series (Daily)"]
        df = spark.createDataFrame([{"timeseries": data}])
       #generic fails 
    except requests.exceptions.RequestException as e:
        print(f"Request failed for {ticker}: {e}")
        return None
    except ValueError as e:  # covers JSON decode errors
        print(f"Unexpected response for {ticker}: {e}")
        return None
    df = df.select(F.explode("timeseries").alias("date", "daily data"))
    company_df = (df
        .select(
            "date",
            F.col("daily data")["1. open"].cast("double").alias("open"),
            F.col("daily data")["2. high"].cast("double").alias("high"),
            F.col("daily data")["3. low"].cast("double").alias("low"),
            F.col("daily data")["4. close"].cast("double").alias("close"),
            F.col("daily data")["5. volume"].cast("long").alias("volume"),
        )
    .withColumn("symbol", F.lit(f"{ticker}"))
    )
    return company_df


## APPLE: AAPL

In [0]:
aapl = api_request("AAPL")
aapl.write.mode("overwrite").saveAsTable("aapl")

## GOOGLE: GOOG

In [0]:
googl = api_request("GOOGL")
googl.write.mode("overwrite").saveAsTable("goog")

## Inno Holdings: INHD

In [0]:
inhd = api_request("INHD")
inhd.write.mode("overwrite").saveAsTable("inhd")

## FGI Industries: FGI

In [0]:
FGI = api_request("FGI")
FGI.write.mode("overwrite").saveAsTable("FGI")